In [2]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'AppleGothic'    # Mac
# plt.rcParams['font.family'] = 'Malgun Gothic' # Windows
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 로드 완료")

라이브러리 로드 완료


In [3]:
df = pd.read_csv('../../../data/processed/steam_stratified_sample.csv')
print("==== 데이터 로드 완료 ====")
df.shape
df.head(5)

==== 데이터 로드 완료 ====


,appid,name_store,release_date,genres,owners,owners_lower,positive,negative,total_reviews,price_spy,ccu,developers,stratum,is_f2p
0,1432860,Sun Haven,2023-03-10,"['Adventure', 'Casual', 'Indie', 'RPG', 'Simul...","500,000 .. 1,000,000",500000,18523,3933,22456,2499,653,Pixel Sprout Studios,large_high,False
1,1473350,(the) Gnorp Apologue,2023-12-14,"['Casual', 'Indie', 'Simulation', 'Strategy']","200,000 .. 500,000",200000,7849,294,8143,699,289,Myco,large_high,False
2,1993150,轮回修仙路,2023-06-19,"['Adventure', 'Indie', 'RPG', 'Simulation']","200,000 .. 500,000",200000,2355,510,2865,1699,14,烟水寒工作室,large_high,False
3,2527500,MiSide,2024-12-10,"['Adventure', 'Indie', 'RPG', 'Simulation']","1,000,000 .. 2,000,000",1000000,108883,2204,111087,1499,631,AIHASTO,large_high,False
4,1169040,Necesse,2025-10-16,"['Action', 'Adventure', 'Indie', 'RPG']","1,000,000 .. 2,000,000",1000000,15988,1083,17071,974,507,Fair Games ApS,large_high,False


In [4]:
import plotly.express as px
import plotly.graph_objects as go

# 1. 분석 데이터 준비
df_plot = df[df['total_reviews'] >= 10].copy()
df_plot['positive_rate'] = (df_plot['positive'] / df_plot['total_reviews']) * 100

# 2. 산점도 생성
fig = px.scatter(
    df_plot, 
    x="owners_lower", 
    y="positive_rate",
    hover_name="name_store",        # 마우스 올리면 게임 이름 표시
    hover_data=["owners", "total_reviews"], # 추가 정보 표시
    log_x=True,                     # X축 로그 스케일 적용
    opacity=0.5,
    title="판매량(Log Scale) vs 긍정률 인터랙티브 산점도",
    labels={"owners_lower": "판매량 (하한)", "positive_rate": "긍정률 (%)"}
)

# 3. 층화 경계선(Threshold) 추가
fig.add_vline(x=20000, line_dash="dash", line_color="red", annotation_text="20k (Mid)")
fig.add_vline(x=200000, line_dash="dash", line_color="darkred", annotation_text="200k (Large)")

# 4. 레이아웃 조정 (가독성 개선)
fig.update_layout(
    xaxis_type="log",
    template="plotly_white",
    height=600
)

fig.show()

In [5]:
# 2만 장 미만인 게임이 실제로 몇 개인지 확인
small_count = len(df[df['owners_lower'] < 20000])
print(f"판매량 2만 장 미만(Small) 게임 수: {small_count}개")

# 0인 값을 1로 살짝 바꿔서 다시 그려보기 (로그 차트에 표시하기 위함)
df_plot['owners_adj'] = df_plot['owners_lower'].replace(0, 1)

# 이 'owners_adj'를 x축으로 해서 다시 그리면 왼쪽에 뭉쳐있는 점들이 보일 것입니다.


판매량 2만 장 미만(Small) 게임 수: 51개


In [ ]:
# 1. 판매량 구간(Scale) 컬럼 생성
def assign_scale(val):
    if val >= 200000: 
        return 'Large (200k+)'
    elif val >= 20000: 
        return 'Mid (20k-200k)'
    else: 
        return 'Small (0-20k)'

df_plot['scale_group'] = df_plot['owners_lower'].apply(assign_scale)

# 2. 박스플롯 그리기
fig = px.box(
    df_plot, 
    x="scale_group", 
    y="positive_rate",
    color="scale_group",
    category_orders={"scale_group": ["Small (0-20k)", "Mid (20k-200k)", "Large (200k+)"]},
    points="all",          # 개별 게임 점들도 옆에 함께 표시 (매우 중요!)
    hover_name="name_store",
    title="판매량 그룹별 긍정률 분포 비교",
    labels={"scale_group": "판매량 그룹", "positive_rate": "긍정률 (%)"}
)

fig.update_layout(showlegend=False, template="plotly_white")
fig.show()


In [ ]:
large_low = df.loc[(df['name_store'] == 'Dirty Wars: September 11') | (df['name_store'] == 'Eaten by Darkness')]
large_low.head()

,appid,name_store,release_date,genres,owners,owners_lower,positive,negative,total_reviews,price_spy,ccu,developers,stratum,is_f2p
44,2230410,Eaten by Darkness,2023-04-01,"['Action', 'Indie']","500,000 .. 1,000,000",500000,3,7,10,99,0,Black Cardioid Games,large_low,False
52,1743470,Dirty Wars: September 11,2023-09-12,"['Action', 'Adventure', 'Indie', 'Strategy']","200,000 .. 500,000",200000,7,8,15,499,0,"Uglycat Studios, Sudaka Games",large_low,False


In [ ]:
# 가격 구간을 임의로 나눕니다 (예: 10$ 미만, 10~30$, 30$ 이상)
def price_category(p):
    if p == 0: 
        return 'Free'
    elif p < 1000: 
        return '< $10'
    elif p < 3000: 
        return '$10 - $30'
    else: return '> $30'

df_plot['price_group'] = df_plot['price_spy'].apply(price_category)

# 층(Stratum)별로 가격 그룹에 따른 긍정률 비교
fig = px.box(
    df_plot, 
    x="scale_group",  # Small, Mid, Large
    y="positive_rate", 
    color="price_group",
    title="층별 가격대(Price Group)에 따른 긍정률 분포",
    points="all",
    hover_name="name_store"
)
fig.show()


In [19]:
import ast

# 1. 문자열 형태의 장르를 실제 리스트로 변환하는 함수
def parse_genres(g):
    try:
        # 이미 리스트라면 그대로 반환, 문자열이라면 리스트로 변환
        if isinstance(g, list): return g
        return ast.literal_eval(g)
    except:
        return []

# 2. 전처리 및 펼치기 (Explode)
df_exploded = df_plot.copy()
df_exploded['temp_genres'] = df_exploded['genres'].apply(parse_genres)
df_exploded = df_exploded.explode('temp_genres')

# 3. 'Indie'와 의미 없는 태그 제외
df_exploded = df_exploded[~df_exploded['temp_genres'].isin(['Indie', 'Early Access', 'Free To Play'])]

# 4. 바이올린 플롯 그리기
fig = px.violin(
    df_exploded, 
    x="temp_genres", 
    y="positive_rate", 
    color="scale_group",
    box=True,
    points="all",
    title="[개별 장르별] 판매량 그룹의 긍정률 분포 비교",
    labels={"temp_genres": "장르", "positive_rate": "긍정률 (%)"},
    category_orders={"scale_group": ["Small (0-20k)", "Mid (20k-200k)", "Large (200k+)"]}
)

fig.update_layout(xaxis={'categoryorder':'total descending'}, template="plotly_white")
fig.show()


In [21]:
# 1. 상관관계 계산을 위한 데이터 준비
# 긍정률이 계산되지 않았다면 여기서 계산
if 'positive_rate' not in df_plot.columns:
    df_plot['positive_rate'] = (df_plot['positive'] / df_plot['total_reviews']) * 100

# 2. 상관계수 계산 (Pearson & Spearman)
pearson_corr = df_plot['owners_lower'].corr(df_plot['positive_rate'], method='pearson')
spearman_corr = df_plot['owners_lower'].corr(df_plot['positive_rate'], method='spearman')

print(f"--- 판매량 vs 긍정률 상관계수 ---")
# print(f"1. Pearson (선형 관계)  : {pearson_corr:.4f}")
print(f"2. Spearman: {spearman_corr:.4f}")

# 3. 추가: 층별(Scale Group)로 나누어 상관계수 확인 (더 세밀한 분석)
print(f"\n--- [규모별] 판매량 vs 긍정률 Spearman 상관계수 ---")
for group in ['Small (0-20k)', 'Mid (20k-200k)', 'Large (200k+)']:
    group_data = df_plot[df_plot['scale_group'] == group]
    if len(group_data) > 1:
        corr = group_data['owners_lower'].corr(group_data['positive_rate'], method='spearman')
        print(f"{group:<15} : {corr:.4f} (표본 수: {len(group_data)})")


--- 판매량 vs 긍정률 상관계수 ---
2. Spearman: 0.0626

--- [규모별] 판매량 vs 긍정률 Spearman 상관계수 ---
Small (0-20k)   : nan (표본 수: 51)
Mid (20k-200k)  : -0.0566 (표본 수: 55)
Large (200k+)   : 0.0326 (표본 수: 54)


In [25]:
# 1. 유료 게임만 추출 (가격이 0보다 큰 데이터)
df_paid = df_plot[df_plot['price_spy'] > 0].copy()

# 2. 상관계수 계산 (Spearman)
spearman_price = df_paid['price_spy'].corr(df_paid['positive_rate'], method='spearman')

print(f"--- [유료 게임] 가격 vs 긍정률 상관계수 ---")
print(f"Spearman: {spearman_price:.4f}")

# 3. 규모별(Scale Group)로 가격의 영향력이 다른지 확인
print(f"\n--- [규모별] 가격 vs 긍정률 Spearman 상관계수 ---")
for group in ['Small (0-20k)', 'Mid (20k-200k)', 'Large (200k+)']:
    group_data = df_paid[df_paid['scale_group'] == group]
    if len(group_data) > 5: # 표본이 어느 정도 있는 경우만
        corr = group_data['price_spy'].corr(group_data['positive_rate'], method='spearman')
        print(f"{group:<15} : {corr:.4f} (표본 수: {len(group_data)})")


--- [유료 게임] 가격 vs 긍정률 상관계수 ---
Spearman: -0.0043

--- [규모별] 가격 vs 긍정률 Spearman 상관계수 ---
Small (0-20k)   : -0.1130 (표본 수: 50)
Mid (20k-200k)  : 0.1058 (표본 수: 53)
Large (200k+)   : -0.0428 (표본 수: 49)


In [ ]:
from scipy import stats

# 유료 게임 전체에 대한 Spearman 상관계수와 P-value
corr, p_value = stats.spearmanr(df_paid['price_spy'], df_paid['positive_rate'])

print(f"--- [전체 유료 게임] 상관관계 검정 ---")
print(f"상관계수: {corr:.4f}")
print(f"P-value : {p_value:.4f}")

if p_value < 0.05:
    print("=> 결과: 통계적으로 유의미한 관계가 있습니다. (p < 0.05)")
else:
    print("=> 결과: 통계적으로 유의미한 관계를 찾을 수 없다.")

--- [전체 유료 게임] 상관관계 검정 ---
상관계수: -0.0043
P-value : 0.9585
=> 결과: 통계적으로 유의미한 관계를 찾을 수 없습니다. (표본 부족 혹은 관계 없음)
